# Create a batch endpoint

In [12]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, AzureCliCredential
import dotenv
import os

dotenv.load_dotenv(override=True)

ml_client = MLClient(
     credential         = AzureCliCredential()
    ,subscription_id    = os.environ["AZURE_SUBSCRIPTION_ID"]
    ,resource_group_name= os.environ["AZURE_RESOURCE_GROUP"]
    ,workspace_name     = os.environ["AZURE_ML"]
)


In [9]:
from azure.ai.ml.entities import BatchEndpoint

endpoint = BatchEndpoint(
     name = "diabetes-batch-endpoint-example"
    ,description = "An example of a batch endpoint"
)

ml_client.batch_endpoints.begin_create_or_update(endpoint)

In [10]:
from azure.ai.ml.entities import BatchDeployment, BatchRetrySettings
from azure.ai.ml.constants import BatchDeploymentOutputAction

registered_model = ml_client.models.get(name="diabetes-model", version=2)

deployment = BatchDeployment(
     name = "diabetes-batch-deployment"
    ,description = "Predictor for diabetes batch predictions"
    ,endpoint_name = endpoint.name
    ,model = registered_model
    ,compute = "diabetes-cluster"
    ,instance_count = 1
    ,max_concurrency_per_instance = 1
    ,mini_batch_size = 2
    ,output_action = BatchDeploymentOutputAction.APPEND_ROW
    ,output_file_name = "predictions.csv"
    ,retry_settings = BatchRetrySettings(max_retries = 3, timeout = 600)
    ,logging_level = "info"
)

ml_client.batch_deployments.begin_create_or_update(deployment)

c:\Users\dgouwy\Documents\devoProjects\azure-machine-learning\.venv_mlflow_skinny\Lib\site-packages\azure\ai\ml\entities\_deployment\batch_deployment.py:137: UserWarning: This class is intended as a base class and it's direct usage is deprecated. Use one of the concrete implementations instead:
* ModelBatchDeployment - For model-based batch deployments
* PipelineComponentBatchDeployment - For pipeline component-based batch deployments
  warnings.warn(


In [11]:
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

input = Input(type=AssetTypes.MLTABLE, path="azureml:diabetes-data-ml-table:0")
job = ml_client.batch_endpoints.invoke(endpoint_name=endpoint.name, inputs={"data":input})

ActivityCompleted: Activity=BatchEndpoint.Invoke, HowEnded=Failure, Duration=2520.39 [ms], Exception=MlException, ErrorCategory=Unknown, ErrorMessage=Tenant mismatch: Token tenant does not match resource tenant


MlException: Tenant mismatch: Token tenant does not match resource tenant